In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path
import subprocess

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

from better_output import show_whole


_ = load_dotenv()

WORKDIR = Path.cwd()

llm = ChatOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    model="deepseek-chat",
    temperature=0.7,
)

def safe_path(p: str) -> Path:
    path = (WORKDIR / p).resolve()
    if not path.is_relative_to(WORKDIR):
        raise ValueError(f"Path escapes workspace: {p}")
    return path

@tool
def run_bash(command: str) -> str:
    """
    Run a shell command.
    """
    dangerous = ["rm -rf /", "sudo", "shutdown", "reboot", "> /dev/"]
    if any(d in command for d in dangerous):
        return "Error: Dangerous command blocked"
    try:
        r = subprocess.run(command, shell=True, cwd=WORKDIR,
                           capture_output=True, text=True, timeout=120,
                           encoding="gbk", errors="replace")
        out = ((r.stdout or "") + (r.stderr or "")).strip()
        return out[:50000] if out else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Timeout (120s)"

@tool
def run_read(path: str, limit: int = None) -> str:
    """
    Read file contents.
    """
    try:
        text = safe_path(path).read_text()
        lines = text.splitlines()
        if limit and limit < len(lines):
            lines = lines[:limit] + [f"...({len(lines) - limit} more lines)"]
        return "\n".join(lines)[:50000]
    except Exception as e:
        return f"Erroe: {e}"

@tool
def run_write(path: str, content: str) -> str:
    """
    Write content to a file.
    """
    try:
        fp = safe_path(path)
        fp.parent.mkdir(parents=True, exist_ok=True)
        fp.write_text(content)
        return f"Wrote {len(content)} bytes to {path}"
    except Exception as e:
        return f"Error: {e}"
    
tools = [run_bash, run_read, run_write]
tool_node = ToolNode(tools)

def assistant(state: MessagesState):
    system_prompt = '你是一个数据分析师和深度学习专家，擅长通过编写Python脚本来实现csv表格数据分析任务，以及构建深度学习模型进行数据预测'
    all_messages = [SystemMessage(system_prompt)] + state['messages']
    model = llm.bind_tools(tools)
    return {'messages': [model.invoke(all_messages)]}

def should_continue(state: MessagesState):
    messages = state['messages']
    last_message = messages[-1]
    if last_message.tool_calls:
        return 'continue'
    return 'end'

builder = StateGraph(MessagesState)

builder.add_node('assistant', assistant)
builder.add_node('tool', tool_node)

builder.add_edge(START, 'assistant')

builder.add_conditional_edges(
    'assistant', 
    should_continue,
    {
        'continue': 'tool',
        'end': END,
    },
)

builder.add_edge('tool', 'assistant')

csv_graph = builder.compile(name='csv-graph')

history = []


while True:
    query = input()
    if query.strip().lower() in ("q", "exit", ""):
        break

    history.append(HumanMessage(content=query))

    response = csv_graph.invoke({'messages': history})

    history = response["messages"]

    for message in reversed(response['messages']):
        if isinstance(message, AIMessage):
            print(f"Assistant: {message.content}\n")
            break


Assistant: 我来先查看一下泰坦尼克号数据集的内容。

